# Breast Cancer Classification Pipeline

This notebook demonstrates the reproducible DS471 workflow from the validated WDBC partitions through baseline and tuned model evaluation. The hold-out test set is used only for final reporting.

## 1. Dataset overview and class distribution

The WDBC dataset contains 569 observations, 30 numerical measurements, and a binary diagnosis target (`M = 1`, `B = 0`).

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

DATA_DIR = ROOT / "data" / "processed"
FIGURE_DIR = ROOT / "reports" / "figures"
TARGET = "diagnosis"
partitions = {name: pd.read_csv(DATA_DIR / f"{name}.csv") for name in ("train", "val", "test")}
train = partitions["train"]
feature_columns = [column for column in train.columns if column != TARGET]
print(f"Train: {train.shape}; features: {len(feature_columns)}")
print("\nClass counts by partition:")
display(pd.DataFrame({name: frame[TARGET].value_counts().sort_index() for name, frame in partitions.items()}).fillna(0).astype(int))
print("\nClass proportions:")
display(pd.DataFrame({name: frame[TARGET].value_counts(normalize=True).sort_index() for name, frame in partitions.items()}).round(4))

## 2. Preprocessing and leakage verification

The preprocessing script fits `StandardScaler` on the training partition only. The persisted files contain the 30 scaled features and the encoded target; the identifier is absent. The checks below confirm the shared schema, finite values, and the expected 70/15/15 partition sizes.

In [ ]:
import numpy as np

assert [len(partitions[name]) for name in ("train", "val", "test")] == [398, 85, 86]
assert "id" not in feature_columns
assert len(feature_columns) == 30
assert all(list(frame.columns) == feature_columns + [TARGET] for frame in partitions.values())
assert all(np.isfinite(frame[feature_columns].to_numpy()).all() for frame in partitions.values())
assert all(not frame.isna().any().any() for frame in partitions.values())
train_mean = train[feature_columns].mean().abs().max()
train_std_error = (train[feature_columns].std(ddof=0) - 1).abs().max()
print("Leakage checks passed.")
print(f"Maximum absolute training mean: {train_mean:.3e}")
print(f"Maximum training population-std error: {train_std_error:.3e}")

## 3. Baseline versus tuned model comparison

Traditional baselines are trained on `train.csv` only. The tuning stage uses stratified five-fold cross-validation over train plus validation, then evaluates the selected estimators once on the untouched test partition.

In [ ]:
from train_traditional import train_and_evaluate
from evaluate_tune import tune_and_evaluate

baseline_results = train_and_evaluate(partitions, feature_columns)
tuned_results = tune_and_evaluate(
    partitions, feature_columns, FIGURE_DIR / "confusion_matrices.png"
)
baseline_test = baseline_results.query("Split == 'test'").copy()
tuned_test = tuned_results.copy()
tuned_test["Model"] = tuned_test["Model"].replace({"MLP": "Multi-Layer Perceptron (MLP)"})
comparison = pd.concat(
    [baseline_test[["Model", "Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]],
     tuned_test[["Model", "Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]]],
    ignore_index=True,
)
display(comparison.set_index("Model").round(4))

The tuned models are selected using cross-validation on the development data. The test metrics provide the final comparison, while recall is particularly important because a false negative can miss a malignant tumor.

## 4. Learning curves and confusion matrices

The figures below summarize the neural-network convergence behavior and the tuned-model error patterns on the hold-out test set.

### MLP learning curves

The training and validation loss curves help identify convergence and possible overfitting during neural-network training.

![MLP learning curves](../reports/figures/mlp_learning_curves.png)

### Tuned-model confusion matrices

These matrices show the number of benign and malignant samples assigned to each predicted class. False negatives (actual malignant, predicted benign) deserve special attention in a screening context.

![Tuned model confusion matrices](../reports/figures/confusion_matrices.png)